# Known Issues Triage — Sandbox

Scratchpad for **acting on** `data/processed/audit/known_issues.csv` produced
by the dataset audit notebook (`data_exploration.ipynb`).

Nine moves, in priority order:

1. Fatal case isolation — 1 row, must be decided before training resumes
2. Worst-offender ranking — cases with multiple warns (highest leverage for relabeling)
3. Issue-code × status cross-tab — does the audit correctly translate issues → `excluded`?
4. Per-category breakdown — are Scoliosis and Normal hit differently?
5. Cobb correlation — do `target_coverage_low` warns cluster on severe cases?
6. Resolution outlier distribution — where does the 512 × 256 resize hurt?
7. Visual triage helper — open image + mask side-by-side for any patient_id
8. Loss weighting hint — per-status metric aggregation template
9. Audit hash receipt — freeze this state for the experiments/ tier

This notebook is **sandbox tier**: no DVC, no rules, personal scratchpad.


## 0 · Setup

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

# Repo root detection
_nb = Path.cwd()
for _p in [_nb, *_nb.parents]:
    if (_p / "ai").is_dir() and (_p / "params.yaml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate repo root")

AUDIT_DIR = REPO_ROOT / "data" / "processed" / "audit"
KNOWN_ISSUES_PATH = AUDIT_DIR / "known_issues.csv"
CLEAN_INDEX_PATH = AUDIT_DIR / "clean_index.csv"
DATA_ROOT = REPO_ROOT / "data" / "raw" / "MaIA_Scoliosis_Dataset"
METRICS_ROOT = DATA_ROOT / "RadiographMetrics"

if not KNOWN_ISSUES_PATH.exists():
    raise FileNotFoundError(
        f"{KNOWN_ISSUES_PATH} missing — run data_exploration.ipynb first."
    )

KNOWN = pd.read_csv(KNOWN_ISSUES_PATH)
CLEAN = pd.read_csv(CLEAN_INDEX_PATH)

print(f"known_issues:  {len(KNOWN)} rows, {KNOWN['patient_id'].nunique()} unique cases")
print(f"clean_index:   {len(CLEAN)} rows")
print(f"issue codes:")
print(KNOWN["issue_code"].value_counts().to_string())
print()
print(f"severity:")
print(KNOWN["severity"].value_counts().to_string())


known_issues:  348 rows, 189 unique cases
clean_index:   250 rows
issue codes:
issue_code
id_out_of_range        214
target_coverage_low     98
resolution_outlier      34
empty_mask               1
cobb_angle_outlier       1

severity:
severity
info     214
warn     133
fatal      1


## 1 · Fatal case isolation

There is exactly **one `fatal` row** in the audit: the case's mask contains
zero target vertebra IDs (6..22), so it cannot contribute to either the
keypoint or segmentation head. Three options:

1. **Drop it** (easiest). The audit already marks it `excluded` in
   `clean_index.csv`, so training is unaffected — but leaving the file on
   disk risks someone re-including it later.
2. **Relabel it** — open the raw mask in an image viewer, check whether
   the labeler used wrong IDs (1..5 or 23..35). If so, remap.
3. **Report in thesis appendix** as a clear drop-reason example.


In [2]:
FATAL = KNOWN[KNOWN["severity"] == "fatal"]
print(FATAL.to_string())

if len(FATAL):
    pid = int(FATAL.iloc[0]["patient_id"])
    cat = FATAL.iloc[0]["category"]
    # Find the row in clean_index — confirm it's marked excluded
    row = CLEAN[(CLEAN["patient_id"] == pid) & (CLEAN["category"] == cat)]
    print(f"\nclean_index status for {cat}/{pid}: {row['status'].values}")
    print(f"image_path:            {row['image_path'].values[0]}")
    print(f"multiclass_mask_path:  {row['multiclass_mask_path'].values[0]}")


     patient_id   category  issue_code severity                                       details
211         107  Scoliosis  empty_mask    fatal  no target vertebra IDs (6..22) found in mask

clean_index status for Scoliosis/107: ['excluded']
image_path:            /home/ortiz/scoliosis/data/raw/MaIA_Scoliosis_Dataset/Scoliosis/S_107.jpg
multiclass_mask_path:  /home/ortiz/scoliosis/data/raw/MaIA_Scoliosis_Dataset/LabelMultiClass_ID_PNG/LabelMulti_S_107.png


## 2 · Worst offenders — cases with ≥2 warns

High leverage — a single relabel fixes multiple issues at once.
Sorted by warn count desc, then by total issue count.

In [3]:
WARNS = KNOWN[KNOWN["severity"] == "warn"]

# Count warns per case (ignore info-level id_out_of_range)
warn_counts = (
    WARNS.groupby(["patient_id", "category"])
    .size()
    .rename("n_warns")
    .reset_index()
)
total_counts = (
    KNOWN.groupby(["patient_id", "category"])
    .size()
    .rename("n_total")
    .reset_index()
)
offenders = warn_counts.merge(total_counts, on=["patient_id", "category"])
offenders = offenders.sort_values(
    ["n_warns", "n_total"], ascending=[False, False]
).reset_index(drop=True)

# Attach the actual issue codes so you can scan them
issue_lists = (
    WARNS.groupby(["patient_id", "category"])["issue_code"]
    .apply(lambda s: ", ".join(sorted(s)))
    .rename("warn_codes")
    .reset_index()
)
offenders = offenders.merge(issue_lists, on=["patient_id", "category"])

# Attach clean_index status
offenders = offenders.merge(
    CLEAN[["patient_id", "category", "status"]],
    on=["patient_id", "category"],
    how="left",
)

print("Top 15 cases by warn count:")
print(offenders.head(15).to_string(index=False))


Top 15 cases by warn count:
 patient_id  category  n_warns  n_total                              warn_codes   status
        107 Scoliosis        2        3 resolution_outlier, target_coverage_low excluded
        141 Scoliosis        2        3 resolution_outlier, target_coverage_low excluded
        137 Scoliosis        2        2 resolution_outlier, target_coverage_low excluded
         22 Scoliosis        1        2                     target_coverage_low excluded
         23 Scoliosis        1        2                     target_coverage_low excluded
         25 Scoliosis        1        2                     target_coverage_low excluded
         26 Scoliosis        1        2                     target_coverage_low excluded
         27 Scoliosis        1        2                     target_coverage_low excluded
         28 Scoliosis        1        2                     target_coverage_low excluded
         29 Scoliosis        1        2                     target_coverage_low ex

## 3 · Issue-code × status cross-tab

Does the audit correctly convert warn/fatal rows into `clean_index`
statuses? A row showing `warn`-severity issue codes paired with
`status=ok` means the audit's cutoff rules let something through.

In [4]:
merged = KNOWN.merge(
    CLEAN[["patient_id", "category", "status"]],
    on=["patient_id", "category"],
    how="left",
)

print("issue_code × status:")
print(pd.crosstab(merged["issue_code"], merged["status"], margins=True))
print()
print("severity × status:")
print(pd.crosstab(merged["severity"], merged["status"], margins=True))


issue_code × status:
status               excluded  ok  warn  All
issue_code                                  
cobb_angle_outlier          0   0     1    1
empty_mask                  1   0     0    1
id_out_of_range            90  60    64  214
resolution_outlier          3   0    31   34
target_coverage_low        98   0     0   98
All                       192  60    96  348

severity × status:
status    excluded  ok  warn  All
severity                         
fatal            1   0     0    1
info            90  60    64  214
warn           101   0    32  133
All            192  60    96  348


## 4 · Per-category breakdown

280 issue rows for Scoliosis vs 68 for Normal. Is Scoliosis
disproportionately polluted, or is it just larger (179 cases vs 71)?

In [5]:
cat_totals = CLEAN.groupby("category").size().rename("n_cases")
issue_totals = KNOWN.groupby("category").size().rename("n_issues")
per_cat = pd.concat([cat_totals, issue_totals], axis=1)
per_cat["issues_per_case"] = per_cat["n_issues"] / per_cat["n_cases"]
print(per_cat)
print()

print("issue_code × category:")
print(pd.crosstab(KNOWN["issue_code"], KNOWN["category"], margins=True))


           n_cases  n_issues  issues_per_case
category                                     
Normal          71        68         0.957746
Scoliosis      179       280         1.564246

issue_code × category:
category             Normal  Scoliosis  All
issue_code                                 
cobb_angle_outlier        0          1    1
empty_mask                0          1    1
id_out_of_range          67        147  214
resolution_outlier        0         34   34
target_coverage_low       1         97   98
All                      68        280  348


## 5 · Correlate `target_coverage_low` with Cobb angle

If coverage-low cases cluster on severe Cobb → vertebrae are off-frame
on bent spines, which is a **labeling/data problem** (crop tighter or
extend the field-of-view).

If coverage-low cases are randomly distributed across the Cobb spectrum →
it's **labeler noise**, fix by inspection.

In [6]:
low_cov = KNOWN[KNOWN["issue_code"] == "target_coverage_low"][
    ["patient_id", "category"]
].drop_duplicates()

joined = low_cov.merge(
    CLEAN[["patient_id", "category", "cobb_angle_deg"]],
    on=["patient_id", "category"], how="left",
)
joined = joined[joined["category"] == "Scoliosis"]  # only Scoliosis has Cobb
joined = joined.dropna(subset=["cobb_angle_deg"])

print(f"target_coverage_low scoliosis cases with Cobb: {len(joined)}")
if len(joined):
    print(joined["cobb_angle_deg"].describe())

# Compare against the population — full scoliosis cohort
pop = CLEAN[
    (CLEAN["category"] == "Scoliosis") & CLEAN["cobb_angle_deg"].notna()
]["cobb_angle_deg"]
print()
print("Full scoliosis population Cobb distribution:")
print(pop.describe())

# Quick rank test — is the coverage-low group skewed high?
try:
    from scipy.stats import mannwhitneyu
    u, p = mannwhitneyu(joined["cobb_angle_deg"], pop, alternative="greater")
    print(f"\nMann-Whitney U (one-sided greater): U={u:.1f}  p={p:.4f}")
    print("(p < 0.05 → coverage-low cases have significantly HIGHER Cobb)")
except ImportError:
    print("\nscipy not available — skip rank test")


target_coverage_low scoliosis cases with Cobb: 97
count    97.000000
mean     61.085334
std      20.315433
min      15.461725
25%      46.998859
50%      61.200684
75%      76.235299
max      90.000000
Name: cobb_angle_deg, dtype: float64

Full scoliosis population Cobb distribution:
count    179.000000
mean      61.198084
std       20.922864
min        3.104491
25%       45.825449
50%       61.969224
75%       76.710746
max       90.000000
Name: cobb_angle_deg, dtype: float64

Mann-Whitney U (one-sided greater): U=8581.5  p=0.5631
(p < 0.05 → coverage-low cases have significantly HIGHER Cobb)


## 6 · Resolution outlier distribution

Training resizes every image to 512 × 256. If the outliers are
*way* larger than the median, the downsample is aggressive → you're
throwing away information. If outliers are *smaller*, the upsample
creates interpolation artifacts.

In [7]:
res_outliers = KNOWN[KNOWN["issue_code"] == "resolution_outlier"]
res_outliers = res_outliers.merge(
    CLEAN[["patient_id", "category", "image_h", "image_w"]],
    on=["patient_id", "category"],
    how="left",
)

print(f"resolution outliers: {len(res_outliers)}")
print()
print("h/w stats on outliers:")
print(res_outliers[["image_h", "image_w"]].describe())
print()
print("h/w stats on full population:")
print(CLEAN[["image_h", "image_w"]].describe())

# Rough aspect-ratio check
res_outliers["aspect"] = res_outliers["image_h"] / res_outliers["image_w"]
print("\noutlier aspect ratios:")
print(res_outliers["aspect"].describe())
print(f"\ntraining aspect ratio (512/256) = {512/256:.2f}")


resolution outliers: 34

h/w stats on outliers:
           image_h      image_w
count    34.000000    34.000000
mean   3316.382353  1531.676471
std     574.772525   474.448881
min    2485.000000   975.000000
25%    2964.750000  1164.250000
50%    3082.000000  1398.000000
75%    3626.750000  1780.750000
max    4999.000000  2972.000000

h/w stats on full population:
           image_h      image_w
count   250.000000   250.000000
mean   1309.616000   519.580000
std     919.388764   487.434011
min     595.000000   123.000000
25%     782.750000   212.250000
50%     919.500000   318.000000
75%    1020.750000   551.750000
max    4999.000000  2972.000000

outlier aspect ratios:
count    34.000000
mean      2.283818
std       0.531104
min       1.513071
25%       1.827756
50%       2.200083
75%       2.634265
max       3.800159
Name: aspect, dtype: float64

training aspect ratio (512/256) = 2.00


## 7 · Visual triage helper

Call `show_case(patient_id, category)` to open image + mask side-by-side.
Useful for quickly inspecting top offenders from Section 2.

In [ ]:
# Case-summary wrapper — cavekit-case-visualization.md R7.
#
# The single source of truth for the six-subplot layout, per-panel
# placeholders, partial-status downgrades, and Cobb derivation lives in
# ``ai/visualization/case_summary.py``. This cell is a thin adapter that
# translates the notebook's ``CLEAN`` / ``KNOWN`` dataframes into the
# inputs ``render_case_summary`` expects, then calls it and displays the
# returned figure. No inline matplotlib panel implementation — that is
# now the library's job.

import csv
import json as _json
import sys as _sys

# The notebook runs from ``notebooks/sandbox/`` and the ``ai`` package is
# not on ``sys.path`` by default. Cell 2 (setup) computed ``REPO_ROOT``;
# reuse it here so ``from ai.visualization import ...`` resolves against
# the in-repo package without requiring a ``pip install -e .``. Idempotent.
if str(REPO_ROOT) not in _sys.path:
    _sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from ai.preprocessing.keypoints import multiclass_mask_to_keypoints
from ai.visualization import CaseAudit, render_case_summary


_SEVERITY_RANK = {"info": 0, "warn": 1, "fatal": 2}


def _nullable_path(value) -> Path | None:
    """Normalise a nullable CLEAN path cell to ``Path`` or ``None``.

    Handles ``None``, ``NaN`` (pandas reads empty cells as float NaN),
    and empty strings uniformly.
    """

    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    s = str(value).strip()
    return Path(s) if s else None


def _load_image(path: Path) -> np.ndarray:
    arr = np.asarray(Image.open(path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr.astype(np.uint8, copy=False)


def _load_mask(path: Path) -> np.ndarray:
    img = Image.open(path)
    if img.mode != "L":
        img = img.convert("L")
    return np.asarray(img).astype(np.uint8, copy=False)


def _load_curve(path: Path | None) -> np.ndarray | None:
    if path is None or not path.exists():
        return None
    try:
        with path.open() as f:
            rows = [
                (float(r["x_px"]), float(r["y_px"]))
                for r in csv.DictReader(f)
            ]
    except (OSError, ValueError, KeyError):
        return None
    return np.asarray(rows, dtype=np.float64) if rows else None


def _load_reported_cobb(path: Path | None) -> float | None:
    if path is None or not path.exists():
        return None
    try:
        with path.open() as f:
            data = _json.load(f)
    except (OSError, _json.JSONDecodeError):
        return None
    value = data.get("cobb_angle_deg")
    try:
        return None if value is None else float(value)
    except (TypeError, ValueError):
        return None


def show_case(patient_id: int, category: str = "Scoliosis") -> dict:
    """Render the full case_summary panel for one audit case.

    Returns the status dict from ``render_case_summary`` so callers can
    inspect ``render_status`` / ``render_notes`` / Cobb fields inline.
    Prints the status line and the per-case KNOWN rows underneath the
    figure so the notebook cell reads like a triage card rather than a
    bare image dump.
    """

    row = CLEAN[
        (CLEAN["patient_id"] == patient_id) & (CLEAN["category"] == category)
    ]
    if row.empty:
        print(f"no row for {category}/{patient_id}")
        return {}
    r = row.iloc[0]

    image = _load_image(Path(r["image_path"]))
    mask = _load_mask(Path(r["multiclass_mask_path"]))
    curve = _load_curve(_nullable_path(r.get("curve_csv_path")))
    cobb_reported = _load_reported_cobb(
        _nullable_path(r.get("metrics_json_path"))
    )
    try:
        keypoints = multiclass_mask_to_keypoints(mask)
    except Exception:
        # Best-effort keypoint derivation — a failure here falls through
        # to a placeholder subplot rather than aborting triage.
        keypoints = None

    mine = KNOWN[
        (KNOWN["patient_id"] == patient_id) & (KNOWN["category"] == category)
    ]
    if mine.empty:
        severity = "info"
        issue_codes: tuple[str, ...] = ()
    else:
        severity = max(mine["severity"], key=lambda s: _SEVERITY_RANK[s])
        issue_codes = tuple(
            c for c in mine["issue_code"].tolist() if c != "id_out_of_range"
        )

    audit = CaseAudit(
        patient_id=int(patient_id),
        category=str(category),
        severity=severity,
        issue_codes=issue_codes,
        cobb_reported_deg=cobb_reported,
    )

    fig, status = render_case_summary(
        image, mask, audit, curve=curve, keypoints=keypoints,
    )
    plt.show()

    rs = status["render_status"]
    rn = status["render_notes"] or "(none)"
    nv = status["n_target_vertebrae"]
    cr = status["cobb_reported_deg"]
    cd = status["cobb_derived_deg"]
    cdl = status["cobb_delta_deg"]
    print(
        f"render_status={rs}  notes={rn}  n_target_vertebrae={nv}  "
        f"cobb_reported={cr}  cobb_derived={cd}  cobb_delta={cdl}"
    )
    if not mine.empty:
        print()
        print(mine[["issue_code", "severity", "details"]].to_string(index=False))
    return status


# Try the fatal case + a cobb_angle_outlier for starters.
print("→ show_case(107, 'Scoliosis')  # fatal empty_mask — expect partial + placeholders")
print("→ show_case(132, 'Scoliosis')  # cobb_angle_outlier (reported cobb=3.1°)")
print()
print("Comment out if you're running headless.")
# show_case(107, "Scoliosis")
# show_case(132, "Scoliosis")


## 8 · Loss-weighting hint — per-status metric aggregation template

Training currently treats `ok` and `warn` identically (both feed the
DataLoader). If you want to report cleaner headline metrics without
throwing away data, **train on both, evaluate split by status**.

The snippet below shows the shape of the join you'd do against a
prediction table produced by the training notebook.

In [9]:
# Synthesize a fake prediction table for illustration.
# Replace with the real per-case Cobb records logged by
# MlflowRun.log_per_case_cobb() in spinenet_architecture.ipynb.

fake_preds = CLEAN[["patient_id", "category", "status", "cobb_angle_deg"]].copy()
fake_preds = fake_preds[fake_preds["category"] == "Scoliosis"].dropna().copy()
rng = np.random.default_rng(42)
fake_preds["pred_cobb"] = fake_preds["cobb_angle_deg"] + rng.normal(0, 5, len(fake_preds))
fake_preds["abs_error"] = (fake_preds["pred_cobb"] - fake_preds["cobb_angle_deg"]).abs()

print("Per-status Cobb MAE (fake predictions):")
print(fake_preds.groupby("status")["abs_error"].agg(["mean", "median", "count"]))
print()
print("Per-status severity confusion will go here once real predictions are wired in.")


Per-status Cobb MAE (fake predictions):
              mean    median  count
status                             
excluded  3.552148  3.194389     97
ok        2.998800  2.765843     10
warn      3.507850  2.491141     72

Per-status severity confusion will go here once real predictions are wired in.


## 9 · Audit hash receipt

Freeze the current audit state. When you promote training into the
`experiments/` tier, re-compute this hash and `assert` it matches to
catch silent dataset drift.

In [10]:
import hashlib

def _file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

print("Audit fingerprint (save this for the experiments/ tier):")
print(f"  known_issues.csv  {_file_sha256(KNOWN_ISSUES_PATH)}")
print(f"  clean_index.csv   {_file_sha256(CLEAN_INDEX_PATH)}")
print()
print("Suggested assertion for experiments notebook:")
print('    assert sha256("data/processed/audit/known_issues.csv") == "..."')


Audit fingerprint (save this for the experiments/ tier):
  known_issues.csv  811bd041073292b0075fcda00adf8f8bdeee8e6f52a82bdf06ce48848992ad21
  clean_index.csv   22effa3be4b19ee75cfde398726240477a4bc7017718b6b6d48b813328919359

Suggested assertion for experiments notebook:
    assert sha256("data/processed/audit/known_issues.csv") == "..."


## Next steps

- If the fatal row inspection (Section 1) surfaces a relabel-able case,
  edit the mask, drop the case from `known_issues.csv`, and re-run the
  audit notebook to regenerate both CSVs.
- If Section 5's Mann-Whitney is significant, document it in the thesis
  and consider a separate evaluation bucket for very-severe cases.
- If Section 2's top offenders share a labeling tool/session, check
  whether a single relabeling pass fixes many at once.
